# ⚖️ Full Pipeline — KAGGLE (2×T4, corpus ~93K điều, LLM fp16)

Sinh `results.json` 2000 câu. Corpus **tự build trên Kaggle** (Phase 0, clone repo). LLM **Qwen2.5-7B fp16 trải đều 2×T4** (không 4-bit) → không OOM.

**Trước khi chạy (Settings bên phải):**
- `Accelerator` → **GPU T4 x2** (bắt buộc cho Phase B — fp16 7B ~15GB cần 2 GPU).
- `Internet` → **On** (clone repo + tải HF model/dataset).
- Upload qua **Add Input**: `stage1_questions.json` (~0.5MB) + **`corpus_emb.npy` của lần chạy trước** (corpus 93K không đổi → khỏi embed lại, tiết kiệm ~1.5-2h). Nhanh nhất: Add Input → **notebook output của phiên trước** (có sẵn cả 2 file).
- ⚠️ **Vòng 4 (CAND=50): XÓA `retrieved.json` cũ** trong `/kaggle/working` trước khi chạy Phase A (file cũ là phễu 20 — cell sẽ skip nếu còn).
- Chạy xong: tải `retrieved.json` (Phase A) / `results.json` + `submission.zip` (Phase B) từ tab **Output**. **Save Version** để giữ giữa các phiên.

## 1. Cài thư viện

In [ ]:
!pip install -q "sentence-transformers>=3.0" "transformers>=4.44" accelerate rank_bm25 datasets

## 2. Kiểm tra GPU

In [ ]:
!nvidia-smi -L

## 3. Phase 0 — Build corpus (~93K điều) trên Kaggle
Clone repo public + stream `tmquan/vbpl-vn` (chỉ giữ Luật/Bộ luật/Nghị định/Thông tư/Pháp lệnh theo chủ đề SME). **Khỏi upload corpus 209MB** — build tại đây bằng internet Kaggle. Bỏ qua nếu `/kaggle/working/corpus_articles.jsonl` đã có.

In [ ]:
# Phase 0 — Build corpus ~93K điều trên Kaggle (clone repo public + stream vbpl-vn). Internet Kaggle nhanh & ổn.
import os, subprocess, shutil
WORK = "/kaggle/working"; CORPUS = f"{WORK}/corpus_articles.jsonl"
if os.path.exists(CORPUS) and os.path.getsize(CORPUS) > 5_000_000:
    print("Đã có corpus:", os.path.getsize(CORPUS)//1024//1024, "MB → BỎ QUA build.")
else:
    get_ipython().system('pip install -q datasets')
    if os.path.isdir("ai_legal_assistant"):
        get_ipython().system('cd ai_legal_assistant && git pull -q')      # lấy config mới nhất
    else:
        get_ipython().system('git clone -q https://github.com/lehuubao2909/ai_legal_assistant.git')
    print("Building corpus (keywords, stream 158K vb ~20-30p, có thể retry SSL — kệ, tự hồi phục)...")
    subprocess.run(["python", "-u", "build_corpus.py", "--mode", "keywords"],
                   cwd="ai_legal_assistant/backend", check=True)
    shutil.copy("ai_legal_assistant/data/corpus_articles.jsonl", CORPUS)
    print("corpus →", CORPUS, "|", os.path.getsize(CORPUS)//1024//1024, "MB")

## 3. Tìm file đầu vào (/kaggle/input hoặc /kaggle/working)

In [ ]:
import os, glob, json
WORK = "/kaggle/working"

def find_file(name):
    cands = [os.path.join(WORK, name), name] + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for p in cands:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Không thấy {name}. Upload qua 'Add Input' hoặc đặt vào /kaggle/working.")

CORPUS_PATH = find_file("corpus_articles.jsonl")
QS_PATH = find_file("stage1_questions.json")
questions = json.load(open(QS_PATH, encoding="utf-8"))
print("corpus:", CORPUS_PATH)
print("questions:", QS_PATH, "→", len(questions), "câu")

## 4. Phase A — Retrieval: lưu **top-20 candidate + điểm rerank (phễu 80)** vào `/kaggle/working/retrieved.json` (cutoff áp ở Phase B → sweep offline được). Bỏ qua nếu đã có.

In [ ]:
import re, numpy as np, torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rank_bm25 import BM25Okapi

EMBED_ID, RERANK_ID = "AITeamVN/Vietnamese_Embedding", "AITeamVN/Vietnamese_Reranker"
# PHỄU SÂU (vòng 8): CAND 50→80, lưu top-20. Thua top-1/2 ở RECALL (ta R0.55 vs họ R0.72);
# recall bị chặn bởi pool top-12 (trần ~0.62) → đào sâu phễu để gold hạng 13-80 lọt vào,
# rồi Phase V (LLM-verify) lọc giữ precision. Rerank 80 cặp/câu (~+15-20p T4), corpus_emb tái dùng.
MAX_SEQ, RERANK_MAX, CAND, CAND_SAVE, RR_BATCH = 1024, 256, 80, 20, 32
RRF_K, W_BM25, W_DENSE = 60, 0.65, 0.35   # RRF có trọng số — ưu BM25 (thuật ngữ luật). Đặt 0.5/0.5 = cũ.
DEV = "cuda" if torch.cuda.is_available() else "cpu"
RET_PATH = f"{WORK}/retrieved.json"
EMB_PATH = f"{WORK}/corpus_emb.npy"
def tok(t): return re.findall(r"\w+", (t or "").lower(), flags=re.UNICODE)

if os.path.exists(RET_PATH):
    cache = {int(k): v for k, v in json.load(open(RET_PATH, encoding="utf-8")).items()}
    print(f"Đã có retrieved.json ({len(cache)} câu) → BỎ QUA Phase A. (Muốn chạy lại CAND=50: xóa file này)")
else:
    corpus = [json.loads(l) for l in open(CORPUS_PATH, encoding="utf-8")
              if l.strip() and json.loads(l).get("doc_number")]
    docs_text = [f"{r['title']}\n{r['text']}" for r in corpus]
    print("corpus:", len(corpus), "| device:", DEV, "| phễu rerank CAND:", CAND)

    emb = SentenceTransformer(EMBED_ID, device=DEV); emb.max_seq_length = MAX_SEQ
    # corpus KHÔNG đổi từ time2 → tái dùng corpus_emb.npy (upload qua Add Input) để khỏi embed 93K (~tiết kiệm 1.5-2h)
    emb_in = EMB_PATH if os.path.exists(EMB_PATH) else (glob.glob("/kaggle/input/**/corpus_emb.npy", recursive=True) or [None])[0]
    if emb_in and os.path.exists(emb_in):
        corpus_emb = np.load(emb_in).astype("float32")
        assert len(corpus_emb) == len(corpus), "corpus_emb.npy không khớp corpus"
        print("Dùng corpus_emb.npy có sẵn → BỎ QUA embed corpus")
    else:
        corpus_emb = emb.encode(docs_text, batch_size=128, normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True).astype("float32")
        np.save(EMB_PATH, corpus_emb); print("Đã lưu", EMB_PATH)
    q_emb = emb.encode([q["question"] for q in questions], batch_size=128, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype("float32")
    bm25 = BM25Okapi([tok(t) for t in docs_text])
    rr_tok = AutoTokenizer.from_pretrained(RERANK_ID)
    rr = AutoModelForSequenceClassification.from_pretrained(RERANK_ID).to(DEV).eval()
    if DEV == "cuda": rr = rr.half()

    @torch.no_grad()
    def rerank(query, idxs):
        pairs = [[query, docs_text[i]] for i in idxs]; out = []
        for j in range(0, len(pairs), RR_BATCH):
            enc = rr_tok(pairs[j:j+RR_BATCH], padding=True, truncation=True,
                         max_length=RERANK_MAX, return_tensors="pt").to(DEV)
            out.extend(rr(**enc).logits.view(-1).float().cpu().tolist())
        return out

    cache = {}
    for n, q in enumerate(questions):
        dense = np.argsort(corpus_emb @ q_emb[n])[::-1][:CAND]
        bm = bm25.get_scores(tok(q["question"])); sp = [i for i in np.argsort(bm)[::-1][:CAND] if bm[i] > 0]
        rrf = {}   # RRF có trọng số: ưu BM25 (số luật/điều/thuật ngữ khớp chính xác)
        for rank, i in enumerate(dense): rrf[int(i)] = rrf.get(int(i), 0) + W_DENSE/(RRF_K+rank+1)
        for rank, i in enumerate(sp):    rrf[int(i)] = rrf.get(int(i), 0) + W_BM25/(RRF_K+rank+1)
        fused = sorted(rrf, key=rrf.get, reverse=True)[:CAND]
        scored = sorted(zip(fused, rerank(q["question"], fused)), key=lambda x: x[1], reverse=True)
        seen, dd = set(), []
        for i, s in scored:
            k = (corpus[i]["doc_number"], corpus[i]["article"])
            if k not in seen: seen.add(k); dd.append((i, s))
        # Lưu top-CAND_SAVE candidate KÈM điểm rerank → cutoff áp ở Phase B (sweep offline được)
        cache[int(q["id"])] = [
            {**{k: corpus[i][k] for k in ("doc_number","clean_name","article","title","text")}, "score": float(s)}
            for i, s in dd[:CAND_SAVE]
        ]
        if (n+1) % 200 == 0: print(f"retrieved {n+1}/{len(questions)}")

    json.dump({str(k): v for k, v in cache.items()}, open(RET_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    print("Đã lưu", RET_PATH, f"(top-{CAND_SAVE} + score, phễu rerank {CAND}) → giải phóng VRAM Phase A")
    del emb, rr, corpus_emb, q_emb; import gc; gc.collect(); torch.cuda.empty_cache()

## 5. Phase B1a — Nạp LLM fp16 trải 2×T4 (CHẠY 1 LẦN; đừng chạy lại để khỏi nạp lại model)

In [ ]:
import gc, os, torch
for _v in ["emb", "rr", "rr_tok", "corpus_emb", "q_emb", "bm25"]:
    globals().pop(_v, None)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoTokenizer, AutoModelForCausalLM
LLM_ID = "Qwen/Qwen2.5-7B-Instruct"   # hợp lệ <14B, Apache. (đổi 3B nếu muốn nhẹ hơn)
ltok = AutoTokenizer.from_pretrained(LLM_ID)
ltok.padding_side = "left"                                   # BẮT BUỘC cho batched generate
if ltok.pad_token is None: ltok.pad_token = ltok.eos_token
# fp16 KHÔNG quant — device_map='auto' tự trải ~15GB lên 2×T4 (no OOM, đủ chất lượng).
# Cần 2 GPU (Accelerator = GPU T4 x2). Nếu chỉ 1 GPU → tràn: đổi model 3B hoặc dùng 4-bit.
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16,
                                           device_map="auto", low_cpu_mem_usage=True)
llm.eval()
print(f"LLM fp16 loaded: {LLM_ID} | {torch.cuda.device_count()} GPU")
print("device_map:", getattr(llm, "hf_device_map", "single"))

## 5b. Phase B1b — Helpers + TEST 5 câu (chạy lại thoải mái, KHÔNG nạp lại model)

In [ ]:
import re, json, glob, os, gc, torch
WORK = globals().get("WORK", "/kaggle/working")
if "cache" not in globals():     # B1b tự nạp nếu chưa chạy Phase A trong session
    _rj = next((p for p in [f"{WORK}/retrieved.json"]
                + sorted(glob.glob("/kaggle/input/**/retrieved.json", recursive=True))
                if os.path.exists(p)), None)
    assert _rj, "Chưa có retrieved.json — chạy Phase A (cell 4) trước."
    cache = {int(k): v for k, v in json.load(open(_rj, encoding="utf-8")).items()}
    print("loaded cache:", len(cache), "from", _rj)
if "questions" not in globals():
    _qj = next((p for p in [f"{WORK}/stage1_questions.json"]
                + sorted(glob.glob("/kaggle/input/**/stage1_questions.json", recursive=True))
                if os.path.exists(p)), None)
    assert _qj, "Chưa có stage1_questions.json"
    questions = json.load(open(_qj, encoding="utf-8"))

# ---- LỌC HIỆU LỰC (leaderboard-verified +0.016~0.027 F2): bỏ luật bị thay thế + dedup version cũ.
SUPERSEDED = {"60/2005/QH11","68/2014/QH13","35/2002/QH10","10/2012/QH13","59/2005/QH11","67/2014/QH13",
              "13/2003/QH11","45/2013/QH13","65/2014/QH13","66/2014/QH13","71/2006/QH11","58/2014/QH13",
              "47/2010/QH12","70/2006/QH11","52/2005/QH11","55/2014/QH13","33/2005/QH11","78/2015/NĐ-CP"}
def _yr(dn):
    m = re.search(r"/((?:19|20)\d{2})/", dn or "") or re.search(r"((?:19|20)\d{2})", dn or "")
    return int(m.group(1)) if m else 0
def drop_superseded(cands):
    if not cands: return cands
    newest = {}
    for d in cands:
        nm = re.sub(r"\s+", " ", (d.get("clean_name") or "").strip().lower())
        if nm: newest[nm] = max(newest.get(nm, 0), _yr(d.get("doc_number", "")))
    out = []
    for d in cands:
        dn = d.get("doc_number", "")
        nm = re.sub(r"\s+", " ", (d.get("clean_name") or "").strip().lower())
        if dn in SUPERSEDED: continue                      # luật chắc chắn bị thay thế
        if nm and _yr(dn) < newest.get(nm, 0): continue    # version cũ hơn cùng tên trong top-12
        out.append(d)
    return out or cands

# ---- CUTOFF — c50_t3m15 = đỉnh leaderboard (ARTICLES_F2 0.5371, P 0.563/R 0.553, phễu 50).
# m1.5 vs m2.0: recall Y HỆT, precision 0.563 vs 0.52 → margin chặt đúng. sib-expand: THUA → bác.
CUT_TOP_K, CUT_MARGIN, CUT_MIN = 3, 1.5, None
def apply_cutoff(cands, top_k=None, margin=None, min_score="_"):
    top_k = CUT_TOP_K if top_k is None else top_k
    margin = CUT_MARGIN if margin is None else margin
    min_score = CUT_MIN if min_score == "_" else min_score
    if not cands: return []
    out = [cands[0]]; top = cands[0].get("score", 0.0)
    for d in cands[1:top_k]:
        s = d.get("score", 0.0)
        if min_score is not None and s < min_score: break
        if margin is not None and s < top - margin: break
        out.append(d)
    return out
def get_ctx(qid): return apply_cutoff(drop_superseded(cache.get(int(qid), [])))

def build_prompt(query, ctx):
    # Cắt text mỗi điều 1800 ký tự: 3 điều ≈ 2300 tokens < 3072 → KHÔNG BAO GIỜ truncate mất
    # phần 'CÂU HỎI' ở cuối prompt (bug cũ: prompt dài bị cắt đuôi → model chép luật thay vì trả lời).
    blocks = [f"[{i}] {d.get('clean_name','')} ({d.get('doc_number','')}) — {d.get('article','')}\n{d.get('text','')[:1800]}"
              for i, d in enumerate(ctx, 1)]
    context_str = "\n\n".join(blocks) if blocks else "(Không có căn cứ.)"
    must = ", ".join(sorted({d.get("article", "") for d in ctx if d.get("article")}))
    system = ("Bạn là trợ lý pháp lý AI cho doanh nghiệp SME Việt Nam. Trả lời DỰA HOÀN TOÀN trên CƠ SỞ PHÁP LÝ; "
              "tuyệt đối không bịa, không suy diễn ngoài căn cứ.\nTốt = chính xác (số liệu/điều kiện/thời hạn lấy từ "
              "điều luật), đầy đủ, thực tiễn, rõ ràng dễ hiểu. Viết HOÀN TOÀN bằng tiếng Việt, không chêm từ tiếng Anh. Trả lời súc tích, có cấu trúc, khoảng 200-350 từ.\nBẮT BUỘC nêu rõ số Điều + tên văn bản cho mỗi căn cứ. "
              "Nếu không đủ căn cứ, nói rõ + khuyến nghị tham vấn luật sư.")
    user = f"CƠ SỞ PHÁP LÝ:\n{context_str}\n\nCÂU HỎI: {query}\n\nTrích dẫn rõ các điều: {must}.\nCâu trả lời:"
    return system, user

def _texts(items):
    out = []
    for q in items:
        s, u = build_prompt(q["question"], get_ctx(q["id"]))
        out.append(ltok.apply_chat_template([{"role": "system", "content": s}, {"role": "user", "content": u}],
                                             add_generation_prompt=True, tokenize=False))
    return out

@torch.no_grad()
def _gen(texts, max_new=600):
    inputs = ltok(texts, return_tensors="pt", padding=True, truncation=True, max_length=3072).to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new, do_sample=False, pad_token_id=ltok.eos_token_id)
    res = [ltok.decode(o, skip_special_tokens=True).strip() for o in out[:, inputs.input_ids.shape[1]:]]
    del inputs, out
    return res

def gen_batch(items, max_new=600):
    """Sinh cả lô; nếu OOM → lùi về từng câu (không bao giờ chết cả lô)."""
    texts = _texts(items)
    try:
        return _gen(texts, max_new)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        res = []
        for t in texts:
            try: res.append(_gen([t], max_new)[0])
            except Exception: res.append(None)
            torch.cuda.empty_cache()
        return res

def build_fields(ctx):
    docs, arts = [], []
    for d in ctx:
        c, nm, a = d.get("doc_number", ""), d.get("clean_name", ""), d.get("article", "")
        if not c or not a: continue
        if f"{c}|{nm}" not in docs: docs.append(f"{c}|{nm}")
        if f"{c}|{nm}|{a}" not in arts: arts.append(f"{c}|{nm}|{a}")
    return docs, arts

def ensure_cit(ans, ctx):
    present = set(re.findall(r"Điều\s+(\d+)", ans)); miss = {}
    for d in ctx:
        m = re.match(r"Điều\s+(\d+)", d.get("article", ""))
        if m and m.group(1) not in present: miss.setdefault(d.get("clean_name", ""), []).append(d["article"])
    if not miss: return ans
    return ans.rstrip() + "\n\nCăn cứ pháp lý áp dụng: " + "; ".join(f"{', '.join(a)} ({nm})" for nm, a in miss.items()) + "."

# TEST 5 câu (in luôn số điều sau cutoff để kiểm tra độ rộng)
for q, ans in zip(questions[:5], gen_batch(questions[:5])):
    ctx = get_ctx(q["id"]); _, ra = build_fields(ctx)
    ans = ensure_cit(ans, ctx) if ctx else "Chưa tìm thấy căn cứ."
    print("=" * 90); print("Q%s (%d điều): %s" % (q["id"], len(ctx), q["question"])); print("Điều:", ra); print(ans)
print(f"\n>>> cutoff c50_t3m15 (top3, margin1.5, phễu 50) + lọc hiệu lực — đỉnh leaderboard 0.5371. Ổn thì chạy Phase B2.")

## 5c. Phase V — LLM-verify top-12 (TÙY CHỌN, ~1.5-2h, checkpoint)
Qwen chấm CÓ/KHÔNG từng candidate: "điều này có căn cứ TRỰC TIẾP trả lời câu hỏi?" → `verified.json`. Tải về + sweep `--verified` → nới recall CÓ KIỂM SOÁT (nới mù đã test: P sập 0.3, F2 0.49). Chạy SAU B1a/B1b. Không chặn B2 — chạy B2 trước hay sau đều được.

In [ ]:
# Phase V — LLM-verify từng candidate top-12 (resume qua verified.json)
import json, os, glob, torch
WORK = globals().get("WORK", "/kaggle/working")
VER = f"{WORK}/verified.json"
qmap = {int(q["id"]): q["question"] for q in questions}

ver = {}
_v0 = next((p for p in [VER] + sorted(glob.glob("/kaggle/input/**/verified.json", recursive=True)) if os.path.exists(p)), None)
if _v0:
    ver = {int(k): v for k, v in json.load(open(_v0, encoding="utf-8")).items()}
    print("resume verify từ:", _v0, f"({len(ver)} câu)")

VB = 24
def _vprompts(qid):
    out = []
    for d in cache.get(qid, []):
        u = (f"CÂU HỎI: {qmap[qid]}\n\nĐIỀU LUẬT: {d.get('clean_name','')} — {d.get('article','')}. {d.get('title','')}\n"
             f"{d.get('text','')[:900]}\n\nĐiều luật trên có chứa căn cứ TRỰC TIẾP để trả lời câu hỏi không? Chỉ trả lời CÓ hoặc KHÔNG.")
        out.append(ltok.apply_chat_template([{"role": "system", "content": "Bạn là chuyên gia thẩm định căn cứ pháp lý. Chỉ trả lời đúng một từ: CÓ hoặc KHÔNG."},
                                             {"role": "user", "content": u}], add_generation_prompt=True, tokenize=False))
    return out

@torch.no_grad()
def _verdicts(texts):
    res = []
    for j in range(0, len(texts), VB):
        inputs = ltok(texts[j:j+VB], return_tensors="pt", padding=True, truncation=True, max_length=1024).to(llm.device)
        out = llm.generate(**inputs, max_new_tokens=4, do_sample=False, pad_token_id=ltok.eos_token_id)
        for o in out[:, inputs.input_ids.shape[1]:]:
            t = ltok.decode(o, skip_special_tokens=True).strip().upper()
            res.append(bool(t.startswith("CÓ") or t.startswith("CO")))
        del inputs, out
    return res

todo_q = [int(q["id"]) for q in questions if int(q["id"]) not in ver]
print("verify còn:", len(todo_q), "câu × ~12 phán quyết")
for n, qid in enumerate(todo_q):
    ver[qid] = _verdicts(_vprompts(qid))
    if (n + 1) % 100 == 0:
        json.dump({str(k): v for k, v in ver.items()}, open(VER, "w", encoding="utf-8"))
        torch.cuda.empty_cache()
        print(f"  verified {n+1}/{len(todo_q)}")
json.dump({str(k): v for k, v in ver.items()}, open(VER, "w", encoding="utf-8"))
ok = sum(sum(v) for v in ver.values()); tot = sum(len(v) for v in ver.values())
print(f"DONE verify: {len(ver)} câu | CÓ: {ok}/{tot} ({100*ok/max(tot,1):.0f}%) → tải verified.json về sweep --verified")

## 6. Phase B2 — FULL 2000 câu (BATCHED, checkpoint mỗi lô, resume). fp16 2×T4 ~2-4h.

In [ ]:
import json, os, glob, torch
RESULTS = f"{WORK}/results.json"
BATCH = 6                                  # max_length 3072 → KV cache lớn; 6 an toàn fp16 2×T4.

results = []
# Resume: ưu tiên bản đang chạy dở ở working; không có → tìm results.json trong /kaggle/input
# (bản dở upload/Add Input từ phiên trước). Lấy bản NHIỀU CÂU NHẤT. Ghi tiếp vào working.
_srcs = ([RESULTS] if os.path.exists(RESULTS) else []) + sorted(glob.glob("/kaggle/input/**/results.json", recursive=True))
for _p in _srcs:
    try:
        _r = json.load(open(_p, encoding="utf-8"))
        if isinstance(_r, list) and len(_r) > len(results):
            results = _r; print(f"resume từ: {_p} ({len(_r)} câu)")
    except Exception:
        pass
done = {r["id"] for r in results}
todo = [q for q in questions if int(q["id"]) not in done]
print(f"resume: {len(done)} xong | còn {len(todo)} | cutoff top_k={CUT_TOP_K} margin={CUT_MARGIN}")

for i in range(0, len(todo), BATCH):
    batch = todo[i:i+BATCH]
    answers = gen_batch(batch)             # tự lùi về từng câu nếu OOM → trả None cho câu lỗi
    for q, ans in zip(batch, answers):
        qid = int(q["id"]); ctx = get_ctx(qid); rd, ra = build_fields(ctx)
        if not ctx:
            ans = "Chưa tìm thấy căn cứ pháp lý phù hợp. Khuyến nghị tham vấn luật sư."
        elif ans is None:
            ans = "Căn cứ pháp lý liên quan: " + "; ".join(ra)
        else:
            ans = ensure_cit(ans, ctx)
        results.append({"id": qid, "question": q["question"], "answer": ans,
                        "relevant_docs": rd, "relevant_articles": ra})
    json.dump(sorted(results, key=lambda r: r["id"]), open(RESULTS, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    torch.cuda.empty_cache()
    print(f"  {len(results)}/{len(questions)}")

results.sort(key=lambda r: r["id"])
json.dump(results, open(RESULTS, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("DONE:", len(results))

## 7. Đóng gói `submission.zip` (tải từ tab Output)

In [ ]:
import zipfile, json, os
RESULTS = f"{WORK}/results.json"
assert len(json.load(open(RESULTS))) == len(questions), "Thiếu câu!"
ZIP = f"{WORK}/submission.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(RESULTS, arcname="results.json")     # flat: results.json ở gốc
print("✓", ZIP, "| tải từ tab Output (Data) bên phải.")

## Ghi chú Kaggle
- **fp16 2×T4 (không quant)**: model trải đều 2 GPU → không OOM. `gen_batch` tự lùi về từng câu nếu OOM → không chết giữa chừng.
- **Session dài** (~12h, 30h GPU/tuần) → 2000 câu (~2-4h) trọn 1 phiên. **Save Version** giữ `retrieved.json`/`corpus_emb.npy`/`results.json` (resume).
- **Cutoff + lọc hiệu lực ở Phase B** (cell 5b): Phase A lưu top-12+score → đổi `CUT_*` không cần chạy lại GPU.
  - Leaderboard journey: 0.317 (32vb) → 0.3877 (cutoff t3m3) → 0.4616 (corpus 93K) → 0.4887 (lọc hiệu lực) → **0.4975 (f_t3m2)**.
  - Chốt **f_t3m2** (top3/margin2): P 0.49 / R 0.519. Sib-expand (kéo điều cùng văn bản): TEST THUA (0.4766, P sập) → bác.
  - `drop_superseded`: bỏ 18 luật bị thay thế + dedup version cũ. Verified +0.016~0.027 F2.
  - **Vòng 4 = CAND 20→50** (mở phễu rerank): chạy lại Phase A (xóa retrieved.json cũ) → sweep lại quanh t3m2.
  - **Sweep offline**: tải `retrieved.json` về local → `python scratch/sweep_cutoff.py` → nộp biến thể (10 bài/ngày).
- **Tại sao không vLLM:** image Kaggle lệch CUDA-build (vllm cu13 vs torch cu12). transformers fp16 chắc ăn.